In [ ]:
import re
import matplotlib.pyplot as plt

# 1. Đọc nội dung file log
file_path = "CVOS_advanced_swin_tiny_6.txt"
with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Khởi tạo các danh sách để lưu dữ liệu training
global_steps = []
losses = []
geo_losses = []
cls_losses = []
accuracies = []
mean_ious = []

# Biến bổ trợ để tính toán Global Step tự động
current_epoch = -1
steps_per_epoch = 0
epoch_base_step = 0

# 2. Sử dụng Regex cấu hình lại để bắt cả số âm (-?)
for line in lines:
    if "Epoch:" in line:
        # Lấy thông tin Epoch và Step trong cặp dấu [] -> Ví dụ: Epoch: [13][50/724]
        epoch_step_match = re.search(r"Epoch:\s*\[(\d+)\]\[(\d+)/(\d+)\]", line)
        if not epoch_step_match:
            continue

        epoch_num = int(epoch_step_match.group(1))
        step_num = int(epoch_step_match.group(2))
        max_steps = int(epoch_step_match.group(3))

        # Tự động tính toán mốc dịch chuyển step khi đổi sang Epoch mới
        if epoch_num != current_epoch:
            current_epoch = epoch_num
            # base_step của epoch hiện tại = Epoch số mấy * tổng số step 1 epoch
            epoch_base_step = current_epoch * max_steps

        # Tính toán Global Step tích lũy tăng dần liên tục
        actual_global_step = epoch_base_step + step_num

        # Regex nâng cấp: Thêm '-?' để nhận diện chính xác các giá trị Loss âm
        loss_match = re.search(r"Loss\s+-?\d+\.\d+\s+\((-?\d+\.\d+)\)", line)
        geo_match = re.search(r"Geo Loss\s+-?\d+\.\d+\s+\((-?\d+\.\d+)\)", line)
        cls_match = re.search(r"Cls Loss\s+-?\d+\.\d+\s+\((-?\d+\.\d+)\)", line)
        acc_match = re.search(r"Accu\s+-?\d+\.\d+\s+\((-?\d+\.\d+)\)", line)
        iou_match = re.search(r"Mean_iou\s+-?\d+\.\d+\s+\((-?\d+\.\d+)\)", line)

        # Kiểm tra điều kiện bóc tách an toàn
        if loss_match and geo_match and cls_match and acc_match and iou_match:
            global_steps.append(actual_global_step)
            losses.append(float(loss_match.group(1)))
            geo_losses.append(float(geo_match.group(1)))
            cls_losses.append(float(cls_match.group(1)))
            accuracies.append(float(acc_match.group(1)))
            mean_ious.append(float(iou_match.group(1)))

# 3. Tiến hành vẽ biểu đồ
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Biểu đồ 1: Biến thiên các hàm Loss
ax1.plot(global_steps, losses, marker="", label="Total Loss", color="red", linewidth=1.5)
ax1.plot(global_steps, geo_losses, marker="", label="Geo Loss", color="orange", linewidth=1.5)
ax1.plot(global_steps, cls_losses, marker="", label="Cls Loss", color="brown", linewidth=1.5)
ax1.set_title("Training Loss Convergence (Continuous Steps)", fontsize=14, fontweight="bold")
ax1.set_xlabel("Global Steps", fontsize=12)
ax1.set_ylabel("Loss Value", fontsize=12)
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend()

# Biểu đồ 2: Biến thiên Accuracy và Mean IoU
ax2.plot(global_steps, accuracies, marker="", label="Accuracy", color="blue", linewidth=1.5)
ax2.plot(global_steps, mean_ious, marker="", label="Mean IoU", color="green", linewidth=1.5)
ax2.set_title("Training Metrics Evaluation (Continuous Steps)", fontsize=14, fontweight="bold")
ax2.set_xlabel("Global Steps", fontsize=12)
ax2.set_ylabel("Score (0.0 - 1.0)", fontsize=12)
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.legend()

plt.tight_layout()
plt.show()

# 4. In ra kết quả Validation cuối cùng
print("=" * 60)
print(f"LỌC THÀNH CÔNG: {len(global_steps)} điểm dữ liệu training.")
print("=" * 60)
print("FINAL VALIDATION METRICS HISTORY:")
for line in lines:
    if "Accu50:" in line:
        print(line.strip())
print("=" * 60)

: 